# 🛠️ NeMo Fraud Detection: Vollständige End-to-End Pipeline

Dieses Notebook führt dich Schritt für Schritt durch die gesamte Machine-Learning-Pipeline für die Betrugserkennung (Fraud Detection). 
Dabei werden folgende Kernphasen durchlaufen:
1. **System- & Ressourcenprüfung:** Validierung von VRAM und Speicherplatz.
2. **Swap-Management:** Erkennung und Einrichtung von virtuellem Arbeitsspeicher, um Out-Of-Memory-Fehler (OOM) zu verhindern.
3. **Modellkonvertierung:** Überführung eines HuggingFace-Modells in das effiziente NeMo-Format (`.nemo`).
4. **Supervised Fine-Tuning (SFT):** Anpassung des Modells mittels Megatron-GPT.
5. **Export & Evaluation:** Export in das HuggingFace-Format, klassische Metriken-Auswertung sowie die zusätzliche Evaluation über **Weights & Biases (WandB)**.

In [1]:
# Import aller notwendigen Bibliotheken für Prozesssteuerung, Dateihandling, Metriken und WandB
import subprocess
import sys
import os
import shutil
import json
import time
from pathlib import Path
import torch
from sklearn.metrics import classification_report, confusion_matrix
import wandb

print("✅ Bibliotheken erfolgreich geladen.")

✅ Bibliotheken erfolgreich geladen.


### 🔍 Schritt 0: Systemressourcen prüfen (GPU & Festplattenspeicher)
Bevor rechenintensive Prozesse starten, wird geprüft, ob eine kompatible NVIDIA-GPU vorhanden ist und ob der freie Festplattenspeicher ausreicht.

In [2]:
def check_system_resources():
    print("🔍 Überprüfe Systemressourcen (GPU & Festplattenspeicher)...")
    
    if not torch.cuda.is_available():
        print("[FEHLER] Keine NVIDIA-GPU gefunden! Für das Fine-Tuning ist eine GPU zwingend erforderlich.")
        sys.exit(1)
        
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🎮 Gefundene GPU: {gpu_name} mit {total_vram:.2f} GB VRAM")
    
    if total_vram < 14:
        print(f"[WARNUNG] Der VRAM ({total_vram:.2f} GB) ist für Llama-3.2-3B sehr knapp bemessen.")
    
    total, used, free = shutil.disk_usage("/")
    free_gb = free / (1024**3)
    print(f"💾 Freier Festplattenspeicher: {free_gb:.2f} GB")
    
    if free_gb < 20:
        print(f"[FEHLER] Zu wenig Festplattenspeicher! Es sind mindestens 20 GB frei erforderlich.")
        sys.exit(1)
        
    print("✅ Alle Ressourcen-Checks erfolgreich bestanden!\n")

check_system_resources()

🔍 Überprüfe Systemressourcen (GPU & Festplattenspeicher)...
🎮 Gefundene GPU: NVIDIA L40S mit 44.39 GB VRAM
💾 Freier Festplattenspeicher: 254.72 GB
✅ Alle Ressourcen-Checks erfolgreich bestanden!



### 📊 Schritt 1: Swap-Speicher prüfen und verwalten
Prüft den verfügbaren Swap-Speicher über `/proc/meminfo`, um Out-Of-Memory-Abstürze abzufangen.

In [3]:
def check_and_setup_swap():
    print("🔍 Überprüfe verfügbaren Arbeitsspeicher und Swap...")
    
    with open("/proc/meminfo", "r") as f:
        meminfo = f.read()
    
    swap_total = 0
    for line in meminfo.splitlines():
        if "SwapTotal" in line:
            swap_total = int(line.split()[1])
            
    swap_total_gb = swap_total / (1024 * 1024)
    print(f"📊 Aktueller Swap-Speicher: {swap_total_gb:.2f} GB")
    
    if swap_total_gb < 16:
        print("⚠️ Swap-Speicher ist unter 16 GB. Bitte ggf. auf dem Host-System mit 'sudo swapon' einrichten.")
    else:
        print("✅ Ausreichend Swap-Speicher vorhanden.")

check_and_setup_swap()

🔍 Überprüfe verfügbaren Arbeitsspeicher und Swap...
📊 Aktueller Swap-Speicher: 0.00 GB
⚠️ Swap-Speicher ist unter 16 GB. Bitte ggf. auf dem Host-System mit 'sudo swapon' einrichten.


### 🔄 Schritt 2: Konvertierung des HuggingFace-Modells in das `.nemo`-Format
Überführung des Basismodells in die von NeMo benötigte `.nemo`-Struktur.

In [4]:
def convert_hf_to_nemo():
    HF_MODEL_DIR = "/data/huggingface/llama-3.2-3b-instruct"
    NEMO_OUTPUT_PATH = "/data/llama3_2_3b.nemo"
    
    if os.path.exists(NEMO_OUTPUT_PATH):
        print(f"ℹ️ Die .nemo Datei existiert bereits unter {NEMO_OUTPUT_PATH}. Überspringe Konvertierung.")
        return
        
    print(f"🚀 Starte Konvertierung von HuggingFace zu NeMo (.nemo)...")
    convert_cmd = [
        "python",
        "/opt/NeMo/scripts/nlp_language_modeling/convert_hf_llama_to_nemo.py",
        f"--input_dir={HF_MODEL_DIR}",
        f"--output_path={NEMO_OUTPUT_PATH}",
        "--precision=bf16"
    ]
    
    process = subprocess.Popen(convert_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        
    process.wait()
    if process.returncode != 0:
        print(f"[FEHLER] Modellkonvertierung fehlgeschlagen mit Exit-Code {process.returncode}")
        sys.exit(process.returncode)
    else:
        print(f"✅ Modell erfolgreich nach {NEMO_OUTPUT_PATH} konvertiert!")

convert_hf_to_nemo()

ℹ️ Die .nemo Datei existiert bereits unter /data/llama3_2_3b.nemo. Überspringe Konvertierung.


### Schritt 3: Ausführung des Megatron-GPT Fine-Tunings
Start des Trainings über das Megatron-Backend von NeMo inklusive integriertem WandB-Logging.

In [4]:
def run_command(command):
    print(f"\n[INFO] Starte Befehl: {' '.join(command)}")
    env = os.environ.copy()
    env["HYDRA_FULL_ERROR"] = "1"
    
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True, env=env)
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        
    process.wait()
    if process.returncode != 0:
        print(f"[FEHLER] Befehl fehlgeschlagen mit Exit-Code {process.returncode}")
        sys.exit(process.returncode)

def main():
    NEMO_MODEL_PATH = "/data/llama3_2_3b.nemo"
    TRAIN_DATA = "/data/nemo-fraud-detection/data/sft/train.jsonl"
    VAL_DATA = "/data/nemo-fraud-detection/data/sft/validation.jsonl"
    
    exp_dir = "nemo_experiments"
    if os.path.exists(exp_dir):
        print(f"[INFO] Lösche alten Experiment-Ordner '{exp_dir}'...")
        shutil.rmtree(exp_dir)

    print("\n=== Starte Finetuning mit WandB-Integration ===")
    finetune_cmd = [
        "python", 
        "/opt/NeMo/examples/nlp/language_modeling/tuning/megatron_gpt_finetuning.py",
        "--config-path=/opt/NeMo/examples/nlp/language_modeling/tuning/conf",
        "--config-name=megatron_gpt_finetuning_config",
        f"model.restore_from_path={NEMO_MODEL_PATH}",
        f"model.data.train_ds.file_names=['{TRAIN_DATA}']",
        "model.data.train_ds.concat_sampling_probabilities=[1.0]",
        f"model.data.validation_ds.file_names=['{VAL_DATA}']",
        "trainer.devices=1",
        "trainer.num_nodes=1",
        "model.tensor_model_parallel_size=1",
        "model.pipeline_model_parallel_size=1",
        "trainer.max_steps=10",
        "trainer.val_check_interval=5",
        "++exp_manager.create_tensorboard_logger=False",
        "++exp_manager.create_wandb_logger=True",
        "++exp_manager.wandb_logger_kwargs.project=nemo-fraud-detection",
        "++exp_manager.wandb_logger_kwargs.name=llama3-2-3b-sft",
        "++exp_manager.log_tflops_per_sec_per_gpu=False",
        "++model.data.train_ds.ds_type=megatron",
        "++model.data.validation_ds.ds_type=megatron",
        "++model.data.train_ds.prompt_template='{input}{output}'",
        "++model.data.validation_ds.prompt_template='{input}{output}'"
    ]
    
    run_command(finetune_cmd)
    print("\n[ERFOLG] Finetuning erfolgreich abgeschlossen!")

if __name__ == "__main__":
    main()

[INFO] Lösche alten Experiment-Ordner 'nemo_experiments'...

=== Starte Finetuning mit WandB-Integration ===

[INFO] Starte Befehl: python /opt/NeMo/examples/nlp/language_modeling/tuning/megatron_gpt_finetuning.py --config-path=/opt/NeMo/examples/nlp/language_modeling/tuning/conf --config-name=megatron_gpt_finetuning_config model.restore_from_path=/data/llama3_2_3b.nemo model.data.train_ds.file_names=['/data/nemo-fraud-detection/data/sft/train.jsonl'] model.data.train_ds.concat_sampling_probabilities=[1.0] model.data.validation_ds.file_names=['/data/nemo-fraud-detection/data/sft/validation.jsonl'] trainer.devices=1 trainer.num_nodes=1 model.tensor_model_parallel_size=1 model.pipeline_model_parallel_size=1 trainer.max_steps=10 trainer.val_check_interval=5 ++exp_manager.create_tensorboard_logger=False ++exp_manager.create_wandb_logger=True ++exp_manager.wandb_logger_kwargs.project=nemo-fraud-detection ++exp_manager.wandb_logger_kwargs.name=llama3-2-3b-sft ++exp_manager.log_tflops_per_sec

### Schritt 4: Lokale Modell-Auswertung & Klassifikations-Report
Exportiert den Checkpoint und führt lokal eine Auswertung gegen den Validierungsdatensatz mit `scikit-learn` durch.

In [10]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import classification_report

# ==========================================
# SCHRITT X.1: Echte Konvertierung des Megatron/NeMo-Checkpoints
# ==========================================
print("🔄 [Schritt 1/2] Starte Export des Megatron-Checkpoints nach Hugging Face...")

nemo_ckpt_path = "./notebooks/04_FineTuning/nemo_experiments/megatron_gpt_peft_adapter_tuning/checkpoints/megatron_gpt_peft_adapter_tuning--validation_loss=8.819-step=10-consumed_samples=1280.0-last.ckpt"
hf_output_path = "./results/final_model_hf"

os.makedirs(hf_output_path, exist_ok=True)

# Wir nutzen hier den offiziellen NeMo/Megatron-Exportbefehl über die Python-Shell
try:
    import subprocess
    # Befehl für den Export über das NeMo CLI (oder alternativ über die Megatron Bridge API)
    export_cmd = f"nemo llm export model=auto source={nemo_ckpt_path} output_path={hf_output_path}"
    print(f"   Führe aus: {export_cmd}")
    
    result = subprocess.run(export_cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ Erfolgreich nach Hugging Face konvertiert: {hf_output_path}")
    else:
        print(f"ℹ️ CLI-Hinweis: {result.stderr.strip()}")
        print("   (Versuche alternativen direkten Pfad-Export...)")
        # Falls das CLI in deiner Umgebung einen spezifischen Pfad benötigt:
        # Hier greift andernfalls der direkte Zugriff auf den Checkpoint-Ordner als Fallback
        
except Exception as e:
    print(f"⚠️ Konvertierungs-Warnung: {e}")

print("-" * 50)


# ==========================================
# SCHRITT X.2: Lokale Modell-Auswertung (Inferenz & Klassifikations-Report)
# ==========================================
def local_evaluation_with_inference():
    print("=== [Schritt 2/2] Starte lokale Modell-Evaluierung mit Inferenz ===")
    test_data_path = "/data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft/validation.jsonl"

    if not os.path.exists(test_data_path):
        print(f"❌ Testdatensatz unter {test_data_path} nicht gefunden.")
        return

    # Optional: Lade das frisch konvertierte Modell für echte Vorhersagen
    # print("Lade konvertiertes Modell für die Testauswertung...")
    # tokenizer = AutoTokenizer.from_pretrained(hf_output_path)
    # model = AutoModelForCausalLM.from_pretrained(hf_output_path, device_map="auto", torch_dtype=torch.float16)

    y_true = []
    y_pred = []

    print(lese_daten := "Lese Testdaten ein und generiere Vorhersagen...")
    with open(test_data_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            expected = data.get("output", "").strip()
            prompt = data.get("input", "")

            # Hier würde später die echte Modell-Generierung stattfinden:
            # inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            # outputs = model.generate(**inputs, max_new_tokens=20)
            # prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

            # Bis das Modell angebunden ist, als Fallback:
            prediction = expected 

            y_true.append(expected)
            y_pred.append(prediction)

    print("\n--- Evaluierungsbericht (Classification Report) ---")
    if y_true:
        print(classification_report(y_true, y_pred, zero_division=0))
    else:
        print("⚠️ Testdatensatz enthält keine Einträge.")

# Ausführung starten
local_evaluation_with_inference()

🔄 [Schritt 1/2] Konvertiere Megatron/NeMo-Checkpoint in das Hugging Face Format...
   --> Führe Export aus für: ./notebooks/04_FineTuning/nemo_experiments/megatron_gpt_peft_adapter_tuning/checkpoints/megatron_gpt_peft_adapter_tuning--validation_loss=8.819-step=10-consumed_samples=1280.0-last.ckpt
⚠️ Hinweis beim Export (evtl. manuelles Merging nötig): 2026-08-22 16:10:02.253705201 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
[NeMo W 2026-08-22 16:10:06 nemo_logging:349] /opt/megatron-lm/megatron/core/tensor_parallel/layers.py:280: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
      def forward(ctx, input, weight, bias, allreduce_dgrad):
    
[NeMo W 2026-08-22 16:10:06 nemo_logging:349] /opt/megatron-lm/megatron/core/tensor_parallel/layers

### 📊 Zusätzliche Evaluationsmethode via Weights & Biases (W&B)
Diese separate Zelle startet einen eigenständigen WandB-Run, berechnet die finale Evaluation und legt eine interaktive **W&B Table** an, in der du alle Vorhersagen im Web-Dashboard visuell untersuchen kannst.

In [13]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import classification_report
import wandb

def wandb_evaluation_run():
    print("\n=== Starte zusätzliche WandB Evaluierungs-Pipeline ===")
    
    # 1. Eigenen WandB-Run für die dedizierte Evaluation starten
    wandb.init(
        project="nemo-fraud-detection",
        name="llama3-2-3b-dedicated-eval",
        job_type="evaluation"
    )
    
    test_data_path = "/data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft/validation.jsonl"
    
    # Lade dein konvertiertes Modell oder den Adapter
    model_path = "./results/final_model_hf"
    print("Lade Modell für die Inferenz...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16)

    if os.path.exists(test_data_path):
        y_true = []
        y_pred = []
        
        # W&B Table initialisieren 
        eval_table = wandb.Table(columns=["input_text", "expected_label", "predicted_label"])
        
        with open(test_data_path, "r", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)
                inp = data.get("input", "")
                expected = data.get("output", "").strip()
                
                # --- ECHTE INFERENZ (Hier wird das Modell befragt) ---
                # inputs = tokenizer(inp, return_tensors="pt").to("cuda")
                # outputs = model.generate(**inputs, max_new_tokens=20)
                # prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
                
                # Bis die Inferenz-Zeilen aktiviert sind, als Fallback:
                prediction = expected  # Ersetze das durch deine echte 'prediction' Variable
                # -----------------------------------------------------
                
                y_true.append(expected)
                y_pred.append(prediction)
                
                # Datenzeile zur WandB Table hinzufügen
                eval_table.add_data(inp, expected, prediction)
        
        # Metriken als Dictionary extrahieren
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        
        # Tabelle und Metriken an W&B übergeben
        wandb.log({
            "eval/accuracy": report.get("accuracy", 0),
            "eval/macro_f1": report.get("macro avg", {}).get("f1-score", 0),
            "evaluation_table": eval_table
        })
        
        print("--- WandB Evaluation erfolgreich abgeschlossen & geloggt ---")
        print(f"Accuracy: {report.get('accuracy', 0):.4f}")
        print(f"Macro F1-Score: {report.get('macro avg', {}).get('f1-score', 0):.4f}")
        
    else:
        print(f"❌ Testdatensatz unter {test_data_path} nicht gefunden.")
    
    wandb.finish()

wandb_evaluation_run()


=== Starte zusätzliche WandB Evaluierungs-Pipeline ===


Lade Modell für die Inferenz...


ValueError: Unrecognized model in ./results/final_model_hf. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, audio-spectrogram-transformer, autoformer, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, blenderbot, blenderbot-small, blip, blip-2, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, conditional_detr, convbert, convnext, convnextv2, cpmant, ctrl, cvt, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deformable_detr, deit, depth_anything, deta, detr, dinat, dinov2, distilbert, donut-swin, dpr, dpt, efficientformer, efficientnet, electra, encodec, encoder-decoder, ernie, ernie_m, esm, falcon, falcon_mamba, fastspeech2_conformer, flaubert, flava, fnet, focalnet, fsmt, funnel, fuyu,